# Collaboration network

Default paths are relative: inputs under `data/`, results under `outputs/04_collaboration_network/`. Inputs retain the source private directory layout. Generated tables and figures go to the configured results directory and are not included with the code. No author names, IDs or raw data rows are displayed. The network uses the source saved-vote coverage; node AI percentages use all matched saved decisions as their denominator. Communities are connected components of qualifying coauthorship edges, not a modularity partition.

Coauthor pairs are unordered and count distinct papers. This corrects the source row-order and duplicate-link bugs. The main threshold is eight papers. Existing threshold sensitivity analyses are retained. Affiliation-pair tests assume independence despite shared authors and communities; missing countries retain the source modal-consistency handling, while pairwise overlap excludes missing countries.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '04_collaboration_network'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for group in ['pred','norm']:
    (OUTPUT_DIR/group).mkdir(exist_ok=True)

In [ ]:
import pandas as pd
from collections import Counter
from itertools import combinations
import scipy.stats as stats
from scipy.stats import mannwhitneyu

import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.colors as mcolors
import numpy as np 
import pandas as pd

In [ ]:
good_data = pd.read_csv(str(DATA_DIR / 'norm/dataset/csv/paper_authors.csv'))
vote_result_good = pd.read_csv(str(DATA_DIR / 'norm/dataset/csv/vote_results.csv'))
pred_data = pd.read_csv(str(DATA_DIR / 'pred/dataset/csv/paper_authors.csv'))
vote_result_pred = pd.read_csv(str(DATA_DIR / 'pred/dataset/csv/vote_results.csv'))
for table in [good_data,pred_data]:
    if table[['paper_id','author_id']].isna().any().any():
        raise ValueError('Paper-author identifiers must be nonmissing.')
good_data = good_data.drop_duplicates(['paper_id','author_id'])
pred_data = pred_data.drop_duplicates(['paper_id','author_id'])

In [ ]:
merged_data_pred = pred_data.merge(vote_result_pred, on ="paper_id", validate="many_to_one")

In [ ]:
merged_data_good = good_data.merge(vote_result_good, on ="paper_id", validate="many_to_one")

<h3>Filter and Graph</h3>

In [ ]:
from collections import Counter

def count_exact_author_groups(df):
    """
    Counts how many times each exact group of authors collaborated together.
    """
    paper_authors = df.groupby('paper_id')['author_id'].apply(set)

    # Step 2: Use frozenset to make author groups hashable and countable
    group_counter = Counter(frozenset(authors) for authors in paper_authors)

    group_df = pd.DataFrame([
        {'author_group': list(group), 'group_size': len(group), 'count': count}
        for group, count in group_counter.items()
        if len(group) > 1  # optional: skip solo authors
    ], columns=['author_group','group_size','count']).sort_values(by='count', ascending=False).reset_index(drop=True)

    return group_df

In [ ]:
result = count_exact_author_groups(good_data)
filter_result_good = result[result['count'] > 4]

In [ ]:
print('Normal exact groups with at least five papers:',len(filter_result_good))

In [ ]:
result_pred = count_exact_author_groups(pred_data)
filter_result_pred = result_pred[result_pred['count'] > 4]

In [ ]:
print('Predatory exact groups with at least five papers:',len(filter_result_pred))

<h3>Entire Collaboration Network</h3>

In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def collaboration_network_graph(
    df,
    collaboration_threshold=3,
    filter_type="greater",
    ax=None,
    save_csv=False,
    csv_path=str(OUTPUT_DIR / 'communities_output.csv')
):
    # Extract collaborations (pairwise combinations of authors for each paper)
    collaborations = []
    for _, group in df.groupby("paper_id")["author_id"]:
        collaborations.extend(combinations(sorted(set(group)), 2))

    # Count collaborations
    collaboration_counts = Counter(collaborations)
    
    # Filter collaborations based on the threshold and filter_type
    if filter_type == "greater":
        filtered_collaborations = {
            pair: count for pair, count in collaboration_counts.items() if count >= collaboration_threshold
        }
    elif filter_type == "less":
        filtered_collaborations = {
            pair: count for pair, count in collaboration_counts.items() if 1 < count < collaboration_threshold
        }
    else:
        raise ValueError("Invalid filter_type. Choose 'greater' or 'less'.")
    
    G = nx.Graph()
    for (author1, author2), weight in filtered_collaborations.items():
        G.add_edge(author1, author2, weight=weight)
    
    # Filter authors present in the graph
    filtered_authors = set(G.nodes())
    filtered_data = df[df["author_id"].isin(filtered_authors)]
    
    # Calculate percentage of AI decisions for each author
    ai_percentage = (
        filtered_data.groupby("author_id")["final_decision"]
        .apply(lambda x: (x == "AI").mean() * 100)
        .reset_index(name="ai_percentage")
    )
    ai_percentage_dict = dict(zip(ai_percentage["author_id"], ai_percentage["ai_percentage"]))

    # Calculate node sizes based on the number of papers written
    filtered_author_papers = filtered_data.groupby("author_id")["paper_id"].nunique()
    node_sizes = {
        author: filtered_author_papers.get(author, 0) * 10
        for author in G.nodes()
    }
    
    # Calculate node weights based on collaboration strength
    node_weights = {
        node: sum(d['weight'] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }
    
    # Add AI percentage and node weights as node attributes
    for node in G.nodes():
        G.nodes[node]["ai_percentage"] = ai_percentage_dict.get(node, 0)
        G.nodes[node]["collaboration_weight"] = node_weights[node]
    
    # Authors who can reach each other via the qualifying edge are all in the same community.
    components = [sorted(comp) for comp in nx.connected_components(G)]
    components.sort(key=lambda c: (-len(c), c))

    print(f"detected：{len(components)} communities")
    rows = []
    for i, members in enumerate(components, 1):
        for author in members:
            rows.append({"community_id": i, "author_id": author})

    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(csv_path, index=False)
        print(f"\nfile saved:{csv_path}")
    elif save_csv:
        print("\nfailed")
    
    # Visualize the graph
    created_axis = ax is None
    if created_axis:
        fig, ax = plt.subplots(figsize=(14, 10))
        
    pos = nx.spring_layout(G, seed=42, k=0.8)  # Use spring layout for spacing

    # Normalize AI percentages to a fixed scale of 1–100
    norm = mcolors.Normalize(vmin=1, vmax=100)
    cmap = plt.cm.YlOrRd

    node_colors = [cmap(norm(max(1, min(G.nodes[node].get("ai_percentage", 0), 100)))) for node in G.nodes()]
    
    nx.draw_networkx_nodes(
        G, pos,
        node_size=[node_sizes[node] for node in G.nodes()],
        node_color=node_colors,
        ax=ax,
    )
    nx.draw_networkx_edges(
        G, pos,
        edge_color="gray",
        width=[(G[u][v]["weight"] / max(1, collaboration_threshold)) ** 1.5 for u, v in G.edges()],
        ax=ax,
    )
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, label="Percentage of AI Decisions (%)")
    
    ax.set_title(
        f"Collaboration Network with AI Decision Heatmap\n"
        f"(Threshold: {filter_type} {collaboration_threshold})"
    )

    if created_axis:
        plt.show()

    return G

In [ ]:
collaboration_network_graph(merged_data_good, collaboration_threshold=8, filter_type="greater", save_csv=True, csv_path=str(OUTPUT_DIR / 'norm/communities_output.csv'))

In [ ]:
collaboration_network_graph(merged_data_pred, collaboration_threshold=8, filter_type="greater", save_csv=True, csv_path=str(OUTPUT_DIR / 'pred/communities_output.csv'))

In [ ]:
import matplotlib.pyplot as plt

# Convert width from cm to inches
fig_width = 18.4 / 2.54  # ~7.24 inches
fig_height = 4.5  # adjust based on your desired aspect ratio

fig, axes = plt.subplots(1, 2, figsize=(fig_width, fig_height))

# Plot for Predatory
collaboration_network_graph(merged_data_pred, collaboration_threshold=8, filter_type="greater", ax=axes[0])
axes[0].text(0, 1.05, 'A', transform=axes[0].transAxes,
             fontsize=10, fontweight='bold', va='top', ha='left', family='sans-serif')
axes[0].set_title("Predatory", fontsize=10, family='sans-serif', y=-0.08)

# Plot for Regular
collaboration_network_graph(merged_data_good, collaboration_threshold=8, filter_type="greater", ax=axes[1])
axes[1].text(0, 1.05, 'B', transform=axes[1].transAxes,
             fontsize=10, fontweight='bold', va='top', ha='left', family='sans-serif')
axes[1].set_title("Normal", fontsize=10, family='sans-serif', y=-0.08)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'collaboration_network_grid.png'), format='png', bbox_inches='tight',dpi=300)

plt.show()

In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def collaboration_network_graph(
    df,
    collaboration_threshold=3,
    filter_type="greater",
    ax=None,
    save_csv=False,
    csv_path=str(OUTPUT_DIR / 'communities_output.csv'),
    show_colorbar=True,
    colorbar_fontsize=12
):
    collaborations = []
    for _, group in df.groupby("paper_id")["author_id"]:
        collaborations.extend(combinations(sorted(set(group)), 2))

    collaboration_counts = Counter(collaborations)

    if filter_type == "greater":
        filtered_collaborations = {
            pair: count for pair, count in collaboration_counts.items() if count >= collaboration_threshold
        }
    elif filter_type == "less":
        filtered_collaborations = {
            pair: count for pair, count in collaboration_counts.items() if 1 < count < collaboration_threshold
        }
    else:
        raise ValueError("Invalid filter_type. Choose 'greater' or 'less'.")

    G = nx.Graph()
    for (author1, author2), weight in filtered_collaborations.items():
        G.add_edge(author1, author2, weight=weight)

    filtered_authors = set(G.nodes())
    filtered_data = df[df["author_id"].isin(filtered_authors)]

    ai_percentage = (
        filtered_data.groupby("author_id")["final_decision"]
        .apply(lambda x: (x == "AI").mean() * 100)
        .reset_index(name="ai_percentage")
    )
    ai_percentage_dict = dict(zip(ai_percentage["author_id"], ai_percentage["ai_percentage"]))

    filtered_author_papers = filtered_data.groupby("author_id")["paper_id"].nunique()
    node_sizes = {
        author: filtered_author_papers.get(author, 0) * 10
        for author in G.nodes()
    }

    node_weights = {
        node: sum(d['weight'] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    for node in G.nodes():
        G.nodes[node]["ai_percentage"] = ai_percentage_dict.get(node, 0)
        G.nodes[node]["collaboration_weight"] = node_weights[node]

    components = [sorted(comp) for comp in nx.connected_components(G)]
    components.sort(key=lambda c: (-len(c), c))

    print(f"detected：{len(components)} communities")
    rows = []
    for i, members in enumerate(components, 1):
        for author in members:
            rows.append({"community_id": i, "author_id": author})

    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(csv_path, index=False)
        print(f"\nfile saved:{csv_path}")
    elif save_csv:
        print("\nfailed")

    created_axis = ax is None
    if created_axis:
        fig, ax = plt.subplots(figsize=(14, 10))

    pos = nx.spring_layout(G, seed=42, k=0.8)

    norm = mcolors.Normalize(vmin=1, vmax=100)
    cmap = plt.cm.YlOrRd

    node_colors = [cmap(norm(max(1, min(G.nodes[node].get("ai_percentage", 0), 100)))) for node in G.nodes()]

    nx.draw_networkx_nodes(
        G, pos,
        node_size=[node_sizes[node] for node in G.nodes()],
        node_color=node_colors,
        ax=ax,
    )
    nx.draw_networkx_edges(
        G, pos,
        edge_color="gray",
        width=[(G[u][v]["weight"] / max(1, collaboration_threshold)) ** 1.5 for u, v in G.edges()],
        ax=ax,
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    if show_colorbar:
        cbar = plt.colorbar(sm, ax=ax, label="Percentage of AI Decisions (%)")
        cbar.set_label("Percentage of AI Decisions (%)", fontsize=colorbar_fontsize)
        cbar.ax.tick_params(labelsize=colorbar_fontsize)

    if created_axis:
        plt.show()

    return G


fig_width = 18.4 / 2.54
fig_height = 4.5

fig, axes = plt.subplots(1, 2, figsize=(fig_width, fig_height))

collaboration_network_graph(
    merged_data_pred, collaboration_threshold=8, filter_type="greater",
    ax=axes[0], show_colorbar=False
)
axes[0].text(0, 1.05, 'A', transform=axes[0].transAxes,
             fontsize=13, fontweight='bold', va='top', ha='left', family='sans-serif')
axes[0].set_title("Predatory", fontsize=13, family='sans-serif', y=-0.08)

collaboration_network_graph(
    merged_data_good, collaboration_threshold=8, filter_type="greater",
    ax=axes[1], show_colorbar=True, colorbar_fontsize=12
)
axes[1].text(0, 1.05, 'B', transform=axes[1].transAxes,
             fontsize=13, fontweight='bold', va='top', ha='left', family='sans-serif')
axes[1].set_title("Normal", fontsize=13, family='sans-serif', y=-0.08)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'collaboration_network_grid.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()

import matplotlib.pyplot as plt

fig_width = 18.4 / 2.54
fig_height = 4.5

fig, axes = plt.subplots(1, 2, figsize=(fig_width, fig_height))

collaboration_network_graph(
    merged_data_pred, collaboration_threshold=8, filter_type="greater",
    ax=axes[0], show_colorbar=False
)
axes[0].text(0, 1.05, 'A', transform=axes[0].transAxes,
             fontsize=13, fontweight='bold', va='top', ha='left', family='sans-serif')
axes[0].set_title("Predatory", fontsize=13, family='sans-serif', y=-0.08)

collaboration_network_graph(
    merged_data_good, collaboration_threshold=8, filter_type="greater",
    ax=axes[1], show_colorbar=True, colorbar_fontsize=12
)
axes[1].text(0, 1.05, 'B', transform=axes[1].transAxes,
             fontsize=13, fontweight='bold', va='top', ha='left', family='sans-serif')
axes[1].set_title("Normal", fontsize=13, family='sans-serif', y=-0.08)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'collaboration_network_grid.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()

Check frequent collaborated author's afflication by name

In [ ]:
for group in ['pred','norm']:
    community_path = OUTPUT_DIR/group/'communities_output.csv'
    authors_path = DATA_DIR/group/'metadata'/f'{group}_authors_country_affiliation.csv'
    df_comm = pd.read_csv(community_path)
    df_auth = pd.read_csv(authors_path,usecols=['id','affiliations'])
    df_comm['author_id'] = pd.to_numeric(df_comm.author_id,errors='coerce')
    df_auth['id'] = pd.to_numeric(df_auth.id,errors='coerce')
    df_auth = df_auth.sort_values('affiliations',ascending=False,na_position='last').drop_duplicates('id')
    merged = df_comm.merge(df_auth,left_on='author_id',right_on='id',how='left',validate='many_to_one').drop(columns='id').rename(columns={'affiliations':'Affiliation'})
    merged.to_csv(OUTPUT_DIR/group/'community_with_affiliation.csv',index=False)
    print(group,'community authors:',len(merged),'matched metadata:',int(merged['Affiliation'].notna().sum()))

In [ ]:
import pandas as pd
import re
from itertools import combinations
from collections import Counter

df = pd.read_csv(str(OUTPUT_DIR / 'norm/community_with_affiliation.csv'))

STOPWORDS = set("""
university college dept department school faculty division institute inst laboratory lab center centre hospital clinic
medical med medicine health healthcare engineering science sciences technology tech technological polytechnic poly
national state federal provincial municipal city town county
research teaching academic graduate undergraduate education
of the and & for in on at to from with without by de la le der die das und
usa germany france china india korea uk england
""".split())

def normalize_affil(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_affil(text):
    t = normalize_affil(text)
    return set(tok for tok in t.split() if tok not in STOPWORDS and len(tok) > 2)

df["tokens"] = df["Affiliation"].apply(tokenize_affil)

def jacc(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

# ------- per-community analysis -------
community_results = []
all_sims = []

for cid, g in df.groupby("community_id"):
    tokens = list(g["tokens"])

    # pairwise similarities in this community
    pairs = list(combinations(tokens, 2))
    sims = [jacc(a, b) for a, b in pairs] if pairs else []

    # store for per-community report
    avg_sim = sum(sims)/len(sims) if sims else 0.0
    community_results.append((cid, len(tokens), round(avg_sim, 3)))

    # collect for overall report
    all_sims.extend(sims)

# ------- overall similarity -------
overall_similarity = sum(all_sims) / len(all_sims) if all_sims else 0.0

print("=== Per Community Affiliation Similarity ===")
for cid, n_authors, avg_sim in community_results:
    print(f"Community {cid}: authors={n_authors}, avg_similarity={avg_sim}")

print("\n=== Overall Affiliation Similarity ===")
print(f"Overall Average Similarity Across All Communities = {round(overall_similarity, 3)}")

In [ ]:
import pandas as pd
import re
from itertools import combinations
from collections import Counter
from scipy.stats import ttest_ind, mannwhitneyu

pred_path = str(OUTPUT_DIR / 'pred/community_with_affiliation.csv')
norm_path = str(OUTPUT_DIR / 'norm/community_with_affiliation.csv')

STOPWORDS = set("""
university college dept department school faculty division institute inst laboratory lab center centre hospital clinic
medical med medicine health healthcare engineering science sciences technology tech technological polytechnic poly
national state federal provincial municipal city town county
research teaching academic graduate undergraduate education
of the and & for in on at to from with without by de la le der die das und
usa germany france china india korea uk england
""".split())

def normalize_affil(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_affil(text):
    t = normalize_affil(text)
    return set(tok for tok in t.split() if tok not in STOPWORDS and len(tok) > 2)

def jacc(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

def compute_similarity(path):
    df = pd.read_csv(path)
    df["tokens"] = df["Affiliation"].apply(tokenize_affil)

    all_pair_sims = [] # Similarity of all author pairs (for overall)
    community_summary = [] # Statistics for each community

    for cid, g in df.groupby("community_id"):
        tokens = list(g["tokens"])

        pairs = list(combinations(tokens, 2))
        sims = [jacc(a, b) for a, b in pairs] if pairs else []

        if sims:
            avg_sim = sum(sims) / len(sims)
        else:
            avg_sim = 0.0

        community_summary.append({
            "community_id": cid,
            "n_authors": len(tokens),
            "avg_similarity": avg_sim,
        })

        all_pair_sims.extend(sims)

    # overall similarity
    overall = sum(all_pair_sims) / len(all_pair_sims) if all_pair_sims else 0.0

    return overall, community_summary, all_pair_sims


pred_overall, pred_comms, pred_sims = compute_similarity(pred_path)
norm_overall, norm_comms, norm_sims = compute_similarity(norm_path)

print("\n=== Results ===")
print(f"Pred   overall similarity: {pred_overall:.4f}")
print(f"Norm   overall similarity: {norm_overall:.4f}")

# ============ Statistical Tests ============ #
print("\n=== Statistical Tests ===")

# t-test
t_stat, t_p = ttest_ind(pred_sims, norm_sims, equal_var=False)
print(f"T-test:       t={t_stat:.3f}, p={t_p:.6f}")

# Mann-Whitney U test
u_stat, u_p = mannwhitneyu(pred_sims, norm_sims, alternative="two-sided")
print(f"Mann-Whitney: U={u_stat:.3f}, p={u_p:.6f}")


Check frequent collaborated author's afflication by country

In [ ]:
for group in ['pred','norm']:
    community_path = OUTPUT_DIR/group/'communities_output.csv'
    authors_path = DATA_DIR/group/'metadata'/f'{group}_authors_country_affiliation.csv'
    df_comm = pd.read_csv(community_path)
    df_auth = pd.read_csv(authors_path,usecols=['id','country'])
    df_comm['author_id'] = pd.to_numeric(df_comm.author_id,errors='coerce')
    df_auth['id'] = pd.to_numeric(df_auth.id,errors='coerce')
    df_auth = df_auth.sort_values('country',ascending=False,na_position='last').drop_duplicates('id')
    merged = df_comm.merge(df_auth,left_on='author_id',right_on='id',how='left',validate='many_to_one').drop(columns='id').rename(columns={'country':'Country'})
    merged.to_csv(OUTPUT_DIR/group/'community_with_country.csv',index=False)
    print(group,'community authors:',len(merged),'matched metadata:',int(merged['Country'].notna().sum()))

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

pred_path = str(OUTPUT_DIR / 'pred/community_with_country.csv')
norm_path = str(OUTPUT_DIR / 'norm/community_with_country.csv')

def compute_country_consistency(path):
    df = pd.read_csv(path)

    df["Country"] = df["Country"].fillna("Unknown").astype(str)

    results = []

    for cid, g in df.groupby("community_id"):
        total = len(g)
        counts = g["Country"].value_counts()
        top_count = counts.iloc[0]
        consistency = top_count / total

        results.append({
            "community_id": cid,
            "n_authors": total,
            "top_country": counts.index[0],
            "consistency": consistency
        })

    result_df = pd.DataFrame(results)
    return result_df, result_df["consistency"].tolist()

norm_df, norm_vals = compute_country_consistency(norm_path)
pred_df, pred_vals = compute_country_consistency(pred_path)

print("=== Country Consistency Results ===")
print(f"Norm avg: {sum(norm_vals)/len(norm_vals):.4f}")
print(f"Pred avg: {sum(pred_vals)/len(pred_vals):.4f}")

print("\n=== Statistical Tests ===")

# t-test
t_stat, t_p = ttest_ind(pred_vals, norm_vals, equal_var=False)
print(f"T-test:       t={t_stat:.3f}, p={t_p:.6f}")

# Mann–Whitney U test
u_stat, u_p = mannwhitneyu(pred_vals, norm_vals, alternative="two-sided")
print(f"Mann-Whitney: U={u_stat:.3f}, p={u_p:.6f}")

# print("\nNorm sample:")

# print("\nPred sample:")

<h3> Robustness check: institutional coherence & country overlap across nearby collaboration thresholds (5, 6, 7, 8, 9, 10).<h3>

In [ ]:
import re
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

STOPWORDS = set("""
university college dept department school faculty division institute inst laboratory lab center centre hospital clinic
medical med medicine health healthcare engineering science sciences technology tech technological polytechnic poly
national state federal provincial municipal city town county
research teaching academic graduate undergraduate education
of the and & for in on at to from with without by de la le der die das und
usa germany france china india korea uk england
""".split())


def normalize_affil(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_affil(text):
    t = normalize_affil(text)
    return set(tok for tok in t.split() if tok not in STOPWORDS and len(tok) > 2)


def jacc(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0


def build_communities(df, collaboration_threshold, filter_type="greater"):
    """Same community-detection logic as collaboration_network_graph(),
    with plotting removed. Returns a DataFrame [community_id, author_id]."""
    collaborations = []
    for _, group in df.groupby("paper_id")["author_id"]:
        collaborations.extend(combinations(sorted(set(group)), 2))

    collaboration_counts = Counter(collaborations)

    if filter_type == "greater":
        filtered = {p: c for p, c in collaboration_counts.items() if c >= collaboration_threshold}
    elif filter_type == "less":
        filtered = {p: c for p, c in collaboration_counts.items() if 1 < c < collaboration_threshold}
    else:
        raise ValueError("Invalid filter_type. Choose 'greater' or 'less'.")

    import networkx as nx
    G = nx.Graph()
    for (a1, a2), weight in filtered.items():
        G.add_edge(a1, a2, weight=weight)

    components = [sorted(c) for c in nx.connected_components(G)]
    components.sort(key=lambda c: (-len(c), c))

    rows = []
    for cid, members in enumerate(components, 1):
        for author in members:
            rows.append({"community_id": cid, "author_id": author})
    return pd.DataFrame(rows,columns=['community_id','author_id'])


def institutional_coherence(comm_df, affil_path):
    """Pooled pairwise Jaccard similarity of affiliation tokens within
    each community, pooled across all communities (matches cells 17-19)."""
    df_auth = pd.read_csv(affil_path)
    df_auth["id"] = pd.to_numeric(df_auth["id"], errors="coerce")
    df_auth_sorted = df_auth.sort_values(by=["affiliations"], ascending=False, na_position="last")
    df_auth_dedup = df_auth_sorted.drop_duplicates(subset=["id"], keep="first")

    comm_df = comm_df.copy()
    comm_df["author_id"] = pd.to_numeric(comm_df["author_id"], errors="coerce")
    merged = comm_df.merge(
        df_auth_dedup[["id", "affiliations"]],
        left_on="author_id", right_on="id", how="left",
    ).drop(columns=["id"]).rename(columns={"affiliations": "Affiliation"})
    merged["tokens"] = merged["Affiliation"].apply(tokenize_affil)

    all_pair_sims = []
    for cid, g in merged.groupby("community_id"):
        tokens = list(g["tokens"])
        pairs = list(combinations(tokens, 2))
        sims = [jacc(a, b) for a, b in pairs] if pairs else []
        all_pair_sims.extend(sims)

    overall = float(np.mean(all_pair_sims)) if all_pair_sims else np.nan
    return overall, all_pair_sims


def country_consistency(comm_df, affil_path):
    """Per-community share of authors sharing the modal ROR-derived
    country, averaged (unweighted) across communities (matches cells 21-22)."""
    df_auth = pd.read_csv(affil_path)
    df_auth["id"] = pd.to_numeric(df_auth["id"], errors="coerce")
    df_auth_sorted = df_auth.sort_values(by=["country"], ascending=False, na_position="last")
    df_auth_dedup = df_auth_sorted.drop_duplicates(subset=["id"], keep="first")

    comm_df = comm_df.copy()
    comm_df["author_id"] = pd.to_numeric(comm_df["author_id"], errors="coerce")
    merged = comm_df.merge(
        df_auth_dedup[["id", "country"]],
        left_on="author_id", right_on="id", how="left",
    ).drop(columns=["id"]).rename(columns={"country": "Country"})
    merged["Country"] = merged["Country"].fillna("Unknown").astype(str)

    consistency_vals = []
    for cid, g in merged.groupby("community_id"):
        total = len(g)
        top_count = g["Country"].value_counts().iloc[0]
        consistency_vals.append(top_count / total)

    overall = float(np.mean(consistency_vals)) if consistency_vals else np.nan
    return overall, consistency_vals


PRED_AFFIL_PATH = str(DATA_DIR / 'pred/metadata/pred_authors_country_affiliation.csv')
NORM_AFFIL_PATH = str(DATA_DIR / 'norm/metadata/norm_authors_country_affiliation.csv')

THRESHOLDS = [5, 6, 7, 8, 9, 10]
rows = []

for t in THRESHOLDS:
    comm_pred = build_communities(merged_data_pred, collaboration_threshold=t)
    comm_norm = build_communities(merged_data_good, collaboration_threshold=t)

    coh_pred_overall, coh_pred_vals = institutional_coherence(comm_pred, PRED_AFFIL_PATH)
    coh_norm_overall, coh_norm_vals = institutional_coherence(comm_norm, NORM_AFFIL_PATH)

    ctry_pred_overall, ctry_pred_vals = country_consistency(comm_pred, PRED_AFFIL_PATH)
    ctry_norm_overall, ctry_norm_vals = country_consistency(comm_norm, NORM_AFFIL_PATH)

    # Institutional coherence: Welch t-test + Mann-Whitney (pooled pairwise sims)
    if len(coh_pred_vals) > 1 and len(coh_norm_vals) > 1:
        coh_t, coh_tp = ttest_ind(coh_pred_vals, coh_norm_vals, equal_var=False)
        coh_u, coh_up = mannwhitneyu(coh_pred_vals, coh_norm_vals, alternative="two-sided")
    else:
        coh_t = coh_tp = coh_u = coh_up = np.nan

    # Country consistency: Welch t-test + Mann-Whitney (per-community values)
    if len(ctry_pred_vals) > 1 and len(ctry_norm_vals) > 1:
        ctry_t, ctry_tp = ttest_ind(ctry_pred_vals, ctry_norm_vals, equal_var=False)
        ctry_u, ctry_up = mannwhitneyu(ctry_pred_vals, ctry_norm_vals, alternative="two-sided")
    else:
        ctry_t = ctry_tp = ctry_u = ctry_up = np.nan

    rows.append({
        "threshold": t,
        "n_communities_pred": comm_pred["community_id"].nunique(),
        "n_communities_norm": comm_norm["community_id"].nunique(),
        "coherence_pred": coh_pred_overall,
        "coherence_norm": coh_norm_overall,
        "coherence_t": coh_t, "coherence_p": coh_tp,
        "coherence_mw_p": coh_up,
        "country_pred": ctry_pred_overall,
        "country_norm": ctry_norm_overall,
        "country_t": ctry_t, "country_p": ctry_tp,
        "country_mw_p": ctry_up,
    })

    print(f"\n=== threshold={t} ===")
    print(f"  communities: Pred={comm_pred['community_id'].nunique()}, "
          f"Norm={comm_norm['community_id'].nunique()}")
    print(f"  institutional coherence: Pred={coh_pred_overall:.4f}, Norm={coh_norm_overall:.4f} "
          f"(t={coh_t:.3f}, p={coh_tp:.3g}, MW p={coh_up:.3g})")
    print(f"  country consistency:     Pred={ctry_pred_overall:.4f}, Norm={ctry_norm_overall:.4f} "
          f"(t={ctry_t:.3f}, p={ctry_tp:.3g}, MW p={ctry_up:.3g})")

threshold_robustness_summary = pd.DataFrame(rows)
print("\n\n=== SUMMARY ACROSS THRESHOLDS ===")
print(threshold_robustness_summary.to_string(index=False))

threshold_robustness_summary.to_csv(
    str(OUTPUT_DIR / 'collaboration_threshold_robustness_summary.csv'), index=False
)
print("Threshold summary saved in the external output directory.")

<h3>pairwise country overlap</h3>

In [ ]:
def country_pairwise_overlap(comm_df, affil_path):
    """
    Share of within-community author pairs whose ROR-derived
    affiliation countries are the same.

    Pairs with missing country information are excluded.
    """
    df_auth = pd.read_csv(affil_path)
    df_auth["id"] = pd.to_numeric(df_auth["id"], errors="coerce")

    df_auth_sorted = df_auth.sort_values(
        by=["country"],
        ascending=False,
        na_position="last"
    )
    df_auth_dedup = df_auth_sorted.drop_duplicates(
        subset=["id"],
        keep="first"
    )

    comm_df = comm_df.copy()
    comm_df["author_id"] = pd.to_numeric(
        comm_df["author_id"],
        errors="coerce"
    )

    merged = comm_df.merge(
        df_auth_dedup[["id", "country"]],
        left_on="author_id",
        right_on="id",
        how="left",
    ).drop(columns=["id"]).rename(
        columns={"country": "Country"}
    )

    pair_values = []
    community_values = []

    for cid, g in merged.groupby("community_id"):
        countries = [
            str(c).strip()
            for c in g["Country"]
            if pd.notna(c) and str(c).strip()
        ]

        pairs = list(combinations(countries, 2))
        if not pairs:
            continue

        same = [1.0 if a == b else 0.0 for a, b in pairs]

        # pooled pair-level observations
        pair_values.extend(same)

        # useful if we want a community-level robustness test
        community_values.append(np.mean(same))

    pooled_overlap = (
        float(np.mean(pair_values))
        if pair_values else np.nan
    )

    mean_community_overlap = (
        float(np.mean(community_values))
        if community_values else np.nan
    )

    return pooled_overlap, mean_community_overlap, community_values

In [ ]:
comm_pred = build_communities(merged_data_pred, collaboration_threshold=8)
comm_norm = build_communities(merged_data_good, collaboration_threshold=8)
pair_pred, pair_comm_pred, pair_vals_pred = country_pairwise_overlap(
    comm_pred, PRED_AFFIL_PATH
)
pair_norm, pair_comm_norm, pair_vals_norm = country_pairwise_overlap(
    comm_norm, NORM_AFFIL_PATH
)

if len(pair_vals_pred) > 1 and len(pair_vals_norm) > 1:
    pair_t, pair_tp = ttest_ind(
        pair_vals_pred,
        pair_vals_norm,
        equal_var=False
    )
    pair_u, pair_up = mannwhitneyu(
        pair_vals_pred,
        pair_vals_norm,
        alternative="two-sided"
    )
else:
    pair_t = pair_tp = pair_u = pair_up = np.nan

print(
    f"  pairwise country overlap: "
    f"Pred={pair_pred:.4f}, Norm={pair_norm:.4f} "
    f"(community means: {pair_comm_pred:.4f} vs {pair_comm_norm:.4f}; "
    f"Welch p={pair_tp:.3g}, MW p={pair_up:.3g})"
)